In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\spatial_domain_exp1_placenta.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_all_slides.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
print(df_all.head(4))

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
# Check unique counts and normalized categorical distributions to identify class imbalance
for col in df_all.columns:
    print(df_all[col].value_counts())
    print("")
    print(df_all[col].value_counts(normalize=True))
    print("-"*40)

In [ ]:
all_filenames = df_all["filename"].tolist()
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
# UMAP of tile-level features
import os
from wsidata import open_wsi
import scanpy as sc
import pandas as pd

adatas = []

for i, path in enumerate(all_filenames):
    zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))

    wsi = open_wsi(path, zarr_path)
    adata = wsi.tables["features_conch"]

    match = df_all.loc[df_all["filename"] == path, "rekvnr"]

    if not match.empty:
        rekvnr = str(match.iloc[0])
    else:
        rekvnr = f"unknown_{i}"

    adata.obs["rekvnr"] = [rekvnr] * adata.n_obs
    adatas.append(adata)

adata_concat = sc.concat(
    adatas,
    label="slide_id",
    keys=[f"slide_{i}" for i in range(len(adatas))]
)

adata_concat.obs["rekvnr"] = pd.Categorical(adata_concat.obs["rekvnr"])

In [ ]:
sc.pp.scale(adata_concat)
sc.pp.pca(adata_concat)
sc.pp.neighbors(adata_concat)
sc.tl.umap(adata_concat)

In [ ]:
sc.pl.umap(adata_concat, color='rekvnr', size=3)

In [ ]:
# Compute Spatial Domains Using LazySlide

import lazyslide as zs
import os
from wsidata import open_wsi

models = ['conch', 'h-optimus-0', 'uni']

compute_domains = True

if compute_domains:
    for file in all_filenames:
        zarr_path = os.path.join(zarr_dir, os.path.basename(file).replace(".mrxs", ".zarr"))
        wsi = open_wsi(file, zarr_path)
        
        for model in models:
            feature_key = f'features_{model}'
            domain_key = f'domain_{model}'

            if domain_key in wsi.shapes["tiles_224"]:
                print(f"Key: {domain_key} already exist for slide: {file}")
                continue

            zs.tl.spatial_domain(
                wsi,
                feature_key=feature_key,  
                tile_key="tiles_224",                
                resolution=0.2,                    
                key_added=domain_key                  
            )
            wsi.write(zarr_path)
        
            print(f"Model: {model}, Key added for slide: {file}")

In [ ]:
from spatial_agreement import SpatialAgreement
agreement = SpatialAgreement(all_filenames, zarr_dir, models)

In [ ]:
print(agreement.overall_slide_agreement())

In [ ]:
agreement.summary_slide_agreement()

In [ ]:
i = 0
print(agreement.slide_level_agreement(i))

In [ ]:
agreement.plot_agreement_map(i)

In [ ]:
# UPDATE
from spatial_domain_tile_selection import TileSelector

selector_multi = TileSelector(
        wsi=wsi,
        feature_key="features_conch", 
        domain_keys=["domain_conch", "domain_uni_aligned", "domain_hopt_aligned"],
        tile_key = "tiles_224",
        n_per_domain=4,
        agreement_mode="all_same",
)

selected_tiles_multi = selector_multi.select_tiles_per_domain()
print(f"Selected {len(selected_tiles_multi)} tiles with consensus")
print(selected_tiles_multi.head())

In [ ]:
# UPDATE
selector_multi.plot_domains(selected_tiles_multi)